# Nile-Chat 12B — LoRA Fine-Tuning v2 on Raylab WhatsApp Data (Pass 1 redesign)

A Colab-ready adaptation of `llm_finetuning.ipynb`, updated for the **2026-08-25 Pass 1
redesign**: retrieval-verified narrow examples (real `RetrievalController` classifies
every hypothesis, corrects genuine mismatches), universal brand-filter fidelity
(`metadata_filters` applied for every brand-tagged candidate, matching production
exactly), guaranteed cross-referral seeding across every brand-bearing sheet (not just
Examinations), a tightened grounding gate (phrasing checked against the model's own
JSON, not the wider chunk), and a corrected `CONTEXT:`/`PATIENT MESSAGE:` Alpaca
column mapping (verified against LLaMA-Factory's real `converter.py` source — the
prior dataset trained on an unlabeled, wrong-order `{question}\n{context}` shape).

**Dataset**: 294 gate-accepted → 282 post-dedup → **267 train / 15 val** (smaller than
the original 465/25 baseline — the direct, disclosed cost of the stricter retrieval-
verified sourcing; see Stage 0 below). `FieldSelectionController` is deleted from
production as of this dataset — narrow CONTEXT is now each retrieved chunk's full,
real content, unfiltered, matching what this dataset trains on.

**What you will do here:** load the regenerated dataset from Google Drive, LoRA
fine-tune Nile-Chat 12B **from the original base model** (never continuing from the
previously-merged v1 checkpoint — see the fine-tuning discussion this session: a
second LoRA pass stacked on an already-merged model risks catastrophic forgetting of
the first pass's careful tuning, and LoRA training is cheap enough that there's no
cost reason to avoid a clean restart), evaluate before vs. after, estimate cost/
throughput, serve with vLLM, and load-test.

**What you will NOT do here:** any teacher-distillation or data-formatting work —
that's Stage 0 below, already complete on the source machine.

Runtime: **Colab, 1× A100 GPU** (Runtime → Change runtime type → A100).


## Stage 0 — What's already done (skip these notebook cells)

| Notebook section | Your equivalent | Status |
|---|---|---|
| Tasks + zero-shot Evaluation | `golden_test_suite.json` + collected replies | done |
| Knowledge Distillation (Pass 1 redesign) | `scripts/generate_finetuning_dataset.py` | done — real teacher calls: 449, real cost $8.43 |
| Format Finetuning Datasets | `finetune_data/{grounding_gate,dedup,format_alpaca}.py` | done — 294 gate-accepted → 282 post-dedup (267 train / 15 val) |

Real gate pass rates from the actual full run, for reference: `narrow_positive`
91.7% (133/145), `broad_positive` 94.4% (51/54), `absence`/`ambiguous_direct`/
`ambiguous_cross_chunk` 100%. `format_alpaca.py`'s output was verified directly
against the real `train.json`/`val.json`: 0/282 format violations (every record
correctly `CONTEXT:\n.../PATIENT MESSAGE:\n...`, every output a clean fenced
` ```json ` block + phrasing, zero leaked teacher-scaffolding headers).

Colab's job starts at **Stage 1** below.


## Stage 1 — Colab environment setup

Identical to the proven v1 install — no changes needed here, this stage is not
dataset-specific.


### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')


### Install

Same verified package ranges as the v1 notebook (LLaMA-Factory's real
`check_dependencies()`, the vLLM/torch/torchaudio CUDA-matching fix, the `torchao`
removal, `bitsandbytes` for `adamw_bnb_8bit`) — nothing about the environment setup
changed for this redesign, only the dataset and a couple of config values below.


In [ ]:
# 1. Remove Colab's preinstalled torch stack (and anything from a prior manual
#    install) before vLLM brings in its own matched set -- mixing sources is what
#    causes the torch/torchaudio CUDA mismatch.
!pip uninstall -y torch torchvision torchaudio vllm

# 2. uv is vLLM's own recommended installer for exactly this problem.
!pip install -qU uv

# 3. --torch-backend=auto detects the real CUDA driver on this Colab A100 runtime
#    and installs one mutually-compatible torch/torchvision/torchaudio set for it.
!uv pip install --system vllm --torch-backend=auto

# 4. torchao (pulled in transitively above) breaks peft's LoRA setup below if an
#    old version is present -- removed rather than upgraded (no quantization here).
!pip uninstall -y torchao

# 5. All five exact ranges LLaMA-Factory's own check_dependencies() enforces.
!pip install -qU "transformers>=4.55.0,<=5.8.0,!=4.57.0,!=5.6.0" "datasets>=2.16.0,<=4.0.0" "accelerate>=1.3.0,<=1.15.0" "peft>=0.18.0,<=0.20.0" "trl>=0.18.0,<=0.24.0"

# 6. --no-deps: LLaMA-Factory's own setup.py can otherwise pull a loose/unpinned
#    torch requirement and silently re-upgrade it, undoing step 3's matched set.
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e . --no-deps

# 7. bitsandbytes for the adamw_bnb_8bit optimizer (Stage 5) -- lets us keep
#    cutoff_len at the real full measured value (zero truncation) by cutting
#    optimizer-state memory instead of dataset context.
!pip install -qU "bitsandbytes>=0.50.1"


### Tokens (optional — only if you want W&B logging or a Hub push)

In [ ]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('huggingface')
!huggingface-cli login --token {hf_token}


## Stage 2 — Get your data onto Drive

1. From your machine, upload the contents of
   `D:\Raylab_Project\scripts\finetune_data_out\` — specifically `train.json`,
   `val.json`, `dataset_info.json` — into a **new** Drive folder,
   `/gdrive/MyDrive/raylab-finetune-v2/datasets/`. Deliberately a new folder, not the v1 one — the format
   changed (CONTEXT:/PATIENT MESSAGE: labels, corrected column order) and the record
   count is different (267/15 vs. the old 465/25), so keeping them separate avoids
   ever accidentally training against a stale mix of the two.
2. `dataset_info.json` already uses the correct LLaMA-Factory column mapping
   (`prompt→instruction`, `query→input`, `response→output`, plus `system`/`history`)
   — only the `file_name` paths need updating for Drive, done by the cell below.

(Note: the v1 notebook's own Drive path had a real inconsistency — its markdown said
`raylab-finetune` (hyphen) while its code cell used `raylab_finetune` (underscore).
This notebook uses `raylab-finetune-v2` consistently everywhere below — hyphen, no
exceptions.)


In [ ]:
import json

data_dir = "/gdrive/MyDrive/raylab-finetune-v2/datasets"
info_path = f"{data_dir}/dataset_info.json"

info = json.load(open(info_path, encoding="utf-8"))
info["raylab_finetune_train"]["file_name"] = f"{data_dir}/train.json"
info["raylab_finetune_val"]["file_name"] = f"{data_dir}/val.json"
json.dump(info, open(info_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

!cp {info_path} /content/LLaMA-Factory/data/dataset_info.json
print("dataset_info.json copied into LLaMA-Factory with Drive paths")


## Stage 3 — Baseline sanity check

Loads the **raw base model** (no adapter) — this shows what LoRA is teaching beyond
the model's own zero-shot capability, the same comparison point the v1 run used. If
you also want a baseline against the *currently-deployed* v1 fine-tuned model (a
different, arguably more useful comparison — v2 vs. v1, not v2 vs. raw base), run
this same cell against `mennaharmas/raylab-nilechat-12b` instead and compare both.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model_id = "MBZUAI-Paris/Nile-Chat-12B"

model = AutoModelForCausalLM.from_pretrained(
    base_model_id, device_map="auto", torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Real, resolved whatsapp_mode_a_reply_directive text (Bucket C) -- the exact system
# prompt Mode A uses in production. Re-fetch live via TemplateParser if the directive
# has changed since this was last transcribed.
system_prompt = """انت 'سارة'، موظفة خدمة عملاء مصرية ودودة في مركز رايلاب للأشعة والتحاليل الطبية. اتبع القواعد الآتية بالترتيب ده:

1. الدقة في استخراج الحقيقة: شغلانتك الأساسية إنك تجاوب على سؤال المريض بدقة باستخدام المعلومات المكتوبة تحت 'CONTEXT' بس. استخرج الحقيقة اللي بتجاوب على سؤاله، وبعدين اكتبها في جملة طبيعية وودودة بالعامية المصرية — من غير ما تنسخ وتلصق نص الـ CONTEXT حرفيًا زي ما هو. ممنوع تخترع سعر أو اسم أو أي حقيقة مش موجودة قدامك. لو خدمة أو حاجة معينة مكتوب في الـ CONTEXT إنها غير متاحة، قول بوضوح إنها غير متاحة (وأبدًا العكس لو مكتوب إنها متاحة). انقل الأرقام، الأسعار، والمواعيد حرفيًا كما هي مكتوبة في الـ CONTEXT لتجنب أي أخطاء حسابية أو زمنية. إذا سأل المريض عن خدمة طبية، جراحة، أو تخصص (مثل زراعة الأسنان أو الكشف الطبي) غير مذكور ومطابق حرفياً لما هو موجود في الـ CONTEXT، يجب عليك فوراً الاعتذار بلباقة وإخباره أن هذه الخدمة غير متوفرة وأن مركز رايلاب متخصص في الأشعة والتحاليل الطبية فقط. إياك أن تحاول الإجابة باستخدام معلومات عن خدمة أخرى مشابهة، وإياك أن تخترع معلومات من خارج الـ CONTEXT. ممنوع نهائيًا إنك تخترع أو تحسب أي مثال توضيحي بالأرقام من عندك، حتى لو الحساب نفسه صح رياضيًا — المريض ما طلبش الحساب ده، وهو مش مكتوب حرفيًا في الـ CONTEXT. مثال حرفي على اللي ممنوع تمامًا تعمله: لو الـ CONTEXT بيقول 'النسبه: 0.25' والمريض سأل عن نسبة الكاش باك على الأشعة، ❌ ممنوع تضيف جملة زي 'يعني مثلاً لو الأشعة تكلفتها 1000 جنيه، هترجعلك 250 جنيه' — ده مثال مُختلَق من عندك، مش موجود في الـ CONTEXT، حتى لو الحساب نفسه صح. ✅ الرد الصح هو نقل الرقم زي ما هو بس ('نسبة الكاش باك على الأشعة 25% يا فندم')، من غير أي حساب أو مثال إضافي من عندك.

2. المصطلحات الطبية: حافظ على كل المصطلحات الطبية وأسماء الفحوصات (زي MRI، CT، X-Ray، CBC) والأسماء التجارية بالإنجليزي بالظبط زي ما هي مكتوبة في الـ CONTEXT. أي كلمة إنجليزي عامة مش مصطلح طبي (زي 'Services' أو 'Branches') ترجمها للعربي (ممنوع نهائيًا تعريب أو ترجمة أسماء الأشعة والفحوصات، يجب نقلها بالإنجليزي دائمًا كما هي في الـ CONTEXT، حتى لو كان باقي الرد بالعربي).

3. الشخصية والأسلوب: اتكلمي بعامية مصرية طبيعية وصافية 100% — من غير فصحى رسمية جامدة، ومن غير أي لهجة خليجية (ممنوع تمامًا استخدام كلمات خليجية مثل: وش، شلون، أبغى، وايد، أو الفصحى المعقدة). تحدثي بأسلوب الشارع المصري الراقي والودود.

4. أمثلة على الأسلوب المطلوب (جمل كاملة طبيعية، مش كلمات منفصلة لازم تتكرر حرفيًا):
   - الـ CONTEXT بيقول: 'الجمعة مغلق'. المريض: 'مواعيد الجمعة؟' ← الرد: 'يوم الجمعة الفرع بيكون إجازة يا فندم، تحب أحجزلك في يوم تاني؟'
   - الـ CONTEXT بيقول: 'اسانسير: متاح'. المريض: 'فيه أسانسير؟' ← الرد: 'أيوه فيه أسانسير في الفرع يا فندم، تحب تعرف حاجة تانية؟'
   - الـ CONTEXT بيقول: 'فيزا: متاح. فاليو: غير متاح'. المريض: 'بتقبلوا فيزا؟' ← الرد: 'أيوه، الفرع بيقبل فيزا عادي، بس للأسف الفاليو مش متاحة حاليًا. حابب تعرف طريقة دفع تانية؟'

5. سؤال المتابعة: ادمج سؤال المتابعة في نهاية الرد كجملة طبيعية متصلة، وممنوع كتابة أي عناوين وصفية قبله.

6. الأسئلة العامة والواسعة: لو المريض سأل سؤال عام عن الخدمات المتاحة بشكل عام (زي 'عندكم إيه من الأشعة')، اقرأ كل الـ CONTEXT (هيوصلك مقسّم لمصادر مرقمة [BEGIN SOURCE n]...[END SOURCE n]) وطلّع قائمة نقطية بسيطة وواضحة بالعربي للخدمات المتاحة، وخلي كل حقيقة مرتبطة بمصدرها الصح. خليها مختصرة جدًا.

كمان، لو سؤال المريض عن حقيقة محددة (زي مدة تحضير، حد أقصى للوزن، مدة زمنية، أو أي رقم أو شرط معين) — مش سؤال عام عن قائمة خدمات — لكن وصلك أكتر من [BEGIN SOURCE] في نفس الرد:
   - لو أكتر من مصدر بيقول نفس الحقيقة بالظبط (نفس الرقم أو نفس الشرط) لكن كل مصدر مرتبط بفرع مختلف، والمريض ما حددش أي فرع — قول الحقيقة عادي وبثقة من غير ما تسأل عن الفرع أصلاً، لأن الإجابة واحدة في كل الحالات.
   - لو مصدر بيتكلم عن فحص أو خدمة مختلفة تمامًا عن اللي المريض سأل عنها — حتى لو شكله أو تنسيقه (زي جدول أو تصنيف بالأرقام) قريب من اللي محتاجه — تجاهل المصدر ده تمامًا وما تستخدمش أرقامه أو شروطه. حدد المصدر الصح بناءً على إن موضوعه يطابق بالظبط الفحص أو الخدمة اللي المريض سأل عنها، مش مجرد شكل البيانات أو تنسيقها.
   - ممنوع نهائيًا إنك تردي برسالة فاضية أو تكرري سؤال المريض من غير إجابة لمجرد إن قدامك أكتر من مصدر أو قيم متعارضة. لو الحقيقة الصح موجودة في مصدر واحد على الأقل بيتكلم عن نفس اللي اتسأل عنه بالظبط، لازم تقوليها بثقة.
"""

def generate(system, instruction, input_text):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"{instruction}\n{input_text}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    out = out[:, inputs.input_ids.shape[1]:]
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]


In [ ]:
# Real case from golden_test_suite.json -- swap in whichever case you want to spot-check.
print(generate(
    system_prompt,
    "CONTEXT:\n[Document: Branch Directory] معلومات الفرع الإسعاف: متاح",
    "PATIENT MESSAGE:\nلو حصل طارئ وأنا في فرع حلوان، فيه عربية إسعاف موجودة؟",
))


> **Free the GPU before the next stage needs it.** The model just loaded here
> (~24GB in bf16 for a 12B model) stays resident in this kernel's GPU memory until
> explicitly freed. Stage 5's training subprocess and Stage 9's vLLM server each try
> to load their own full copy on the same GPU.


In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed")


## Stage 4 — Measure `cutoff_len` for real, for THIS dataset

The v1 notebook's real measured numbers (490 records, old format) don't apply here —
the dataset changed (267/15 vs. 465/25) and the format changed (CONTEXT:/PATIENT
MESSAGE: labels are now baked into `instruction`/`input`, adding real tokens per
example that weren't there before). **Run the cell below and use ITS real output** —
don't reuse the v1 percentile table.


In [ ]:
import json

train = json.load(open(f"{data_dir}/train.json", encoding="utf-8"))
lengths = [
    len(tokenizer.encode(r["system"] + r["instruction"] + r["input"] + r["output"]))
    for r in train
]
print("count:", len(lengths))
print("min:", min(lengths), " p50:", sorted(lengths)[len(lengths)//2], " p90:", sorted(lengths)[int(len(lengths)*0.9)])
print("p99:", sorted(lengths)[int(len(lengths) * 0.99)], " max:", max(lengths))


> **Start `cutoff_len` at 4096** (Stage 5's default below), raise it if the measured
> max above exceeds that — the v1 run's own real lesson (documented in Stage 5)
> against truncating the longest, hardest `broad_positive` multi-source examples:
> that trade-off was rejected there in favor of `adamw_bnb_8bit`, and the same
> reasoning applies here if this dataset's real max also exceeds 4096.


## Stage 5 — Configure & run LoRA training

**Trained from the original base model** (`MBZUAI-Paris/Nile-Chat-12B`), never
continuing from the v1 merged checkpoint — a fresh LoRA pass over the combined,
regenerated dataset, per this session's fine-tuning discussion (sequential fine-
tuning on an already-merged model risks catastrophic forgetting of the first pass;
LoRA is cheap enough that there's no cost reason to avoid restarting clean).

Hyperparameters kept identical to the v1 run's proven values (`lora_rank: 16`,
`adamw_bnb_8bit`, `load_best_model_at_end`) — **not** re-tuned for the smaller
dataset, deliberately: no real evidence yet that 267 vs. 465 examples needs a
different rank/epoch count, and changing a proven value without evidence isn't
this project's practice. One real, open number worth watching, not pre-solving:
267 examples / effective batch size 8 ≈ 33 steps/epoch ≈ 100 steps total at 3
epochs — fewer than v1's ~174. Watch `eval_loss` in Stage 6; if it hasn't
plateaued by the last logged step, that's real evidence to raise
`num_train_epochs`, not a reason to raise it preemptively here.

`cutoff_len` below starts at 4096 per Stage 4's guidance — **edit it to match your
own Stage 4 output** before running this cell if your measured max differs.

**Checkpoints save straight to the Hugging Face Hub, not Drive** — `push_to_hub: true` + `hub_model_id: mennaharmas/raylab-nilechat-12b-v2-lora` + `hub_strategy: every_save` (all real `transformers.Seq2SeqTrainingArguments` fields, LLaMA-Factory passes them straight through — verified against its real `TrainingArguments`/`Seq2SeqTrainingArguments` source, not guessed). `output_dir` below is now a local, ephemeral Colab path — that's fine and expected, because the real persistence layer is now the Hub push itself: every time a checkpoint saves (`save_steps: 25`), it's uploaded to the private adapter repo, so a Colab disconnect no longer loses progress the way an un-pushed local-only checkpoint would. Requires Stage 1's `huggingface-cli login` cell to have already run (it has, above) — the Trainer picks up that cached token automatically, no `hub_token` field needed here. Stages 6 and 7 below now load this same adapter from the Hub repo instead of Drive.


In [ ]:
%%writefile /content/LLaMA-Factory/examples/train_lora/raylab_finetune_v2.yaml
### model
model_name_or_path: MBZUAI-Paris/Nile-Chat-12B
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.08
lora_target: all

### dataset
dataset: raylab_finetune_train
eval_dataset: raylab_finetune_val
template: gemma
cutoff_len: 4096  # starting point -- replace with Stage 4's real measured max for THIS dataset if it exceeds 4096
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: /content/raylab-finetune-v2/models/
push_to_hub: true
hub_model_id: mennaharmas/raylab-nilechat-12b-v2-lora
hub_strategy: every_save  # push on every save_steps checkpoint, not just at the end -- real crash-safety
hub_private_repo: true
logging_steps: 5
save_steps: 25  # multiple of eval_steps: 25, required by load_best_model_at_end
plot_loss: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
optim: adamw_bnb_8bit
learning_rate: 1.5e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 25
load_best_model_at_end: true
metric_for_best_model: eval_loss

report_to: wandb
run_name: raylab-nilechat-lora-v2


In [ ]:
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True: real fix from the v1 run's own
# mid-training OOM (allocator fragmentation across this dataset's varying sequence
# lengths). Zero accuracy cost -- purely a memory-allocator setting.
!cd LLaMA-Factory/ && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/raylab_finetune_v2.yaml


## Stage 6 — Evaluate: before vs. after

Load base + adapter (now pulled from the Hub repo `mennaharmas/raylab-nilechat-12b-v2-lora` Stage 5 pushed to, not Drive), reuse the exact same real case from Stage 3 for a genuine before/after.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

base_model_id = "MBZUAI-Paris/Nile-Chat-12B"
adapter_dir = "mennaharmas/raylab-nilechat-12b-v2-lora"  # a Hub repo id -- model.load_adapter() resolves this via
# huggingface_hub the same way it would a local path; no Drive dependency anymore.

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

print("Loading base model in 8-bit...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_id, device_map="auto", quantization_config=quantization_config
)

print("Loading adapter...")
model.load_adapter(adapter_dir)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

system_prompt = """انت 'سارة'، موظفة خدمة عملاء مصرية ودودة في مركز رايلاب للأشعة والتحاليل الطبية. اتبع القواعد الآتية بالترتيب ده:

1. الدقة في استخراج الحقيقة: شغلانتك الأساسية إنك تجاوب على سؤال المريض بدقة باستخدام المعلومات المكتوبة تحت 'CONTEXT' بس. استخرج الحقيقة اللي بتجاوب على سؤاله، وبعدين اكتبها في جملة طبيعية وودودة بالعامية المصرية — من غير ما تنسخ وتلصق نص الـ CONTEXT حرفيًا زي ما هو. ممنوع تخترع سعر أو اسم أو أي حقيقة مش موجودة قدامك. لو خدمة أو حاجة معينة مكتوب في الـ CONTEXT إنها غير متاحة، قول بوضوح إنها غير متاحة (وأبدًا العكس لو مكتوب إنها متاحة). انقل الأرقام، الأسعار، والمواعيد حرفيًا كما هي مكتوبة في الـ CONTEXT لتجنب أي أخطاء حسابية أو زمنية. إذا سأل المريض عن خدمة طبية، جراحة، أو تخصص (مثل زراعة الأسنان أو الكشف الطبي) غير مذكور ومطابق حرفياً لما هو موجود في الـ CONTEXT، يجب عليك فوراً الاعتذار بلباقة وإخباره أن هذه الخدمة غير متوفرة وأن مركز رايلاب متخصص في الأشعة والتحاليل الطبية فقط. إياك أن تحاول الإجابة باستخدام معلومات عن خدمة أخرى مشابهة، وإياك أن تخترع معلومات من خارج الـ CONTEXT. ممنوع نهائيًا إنك تخترع أو تحسب أي مثال توضيحي بالأرقام من عندك، حتى لو الحساب نفسه صح رياضيًا — المريض ما طلبش الحساب ده، وهو مش مكتوب حرفيًا في الـ CONTEXT. مثال حرفي على اللي ممنوع تمامًا تعمله: لو الـ CONTEXT بيقول 'النسبه: 0.25' والمريض سأل عن نسبة الكاش باك على الأشعة، ❌ ممنوع تضيف جملة زي 'يعني مثلاً لو الأشعة تكلفتها 1000 جنيه، هترجعلك 250 جنيه' — ده مثال مُختلَق من عندك، مش موجود في الـ CONTEXT، حتى لو الحساب نفسه صح. ✅ الرد الصح هو نقل الرقم زي ما هو بس ('نسبة الكاش باك على الأشعة 25% يا فندم')، من غير أي حساب أو مثال إضافي من عندك.

2. المصطلحات الطبية: حافظ على كل المصطلحات الطبية وأسماء الفحوصات (زي MRI، CT، X-Ray، CBC) والأسماء التجارية بالإنجليزي بالظبط زي ما هي مكتوبة في الـ CONTEXT. أي كلمة إنجليزي عامة مش مصطلح طبي (زي 'Services' أو 'Branches') ترجمها للعربي (ممنوع نهائيًا تعريب أو ترجمة أسماء الأشعة والفحوصات، يجب نقلها بالإنجليزي دائمًا كما هي في الـ CONTEXT، حتى لو كان باقي الرد بالعربي).

3. الشخصية والأسلوب: اتكلمي بعامية مصرية طبيعية وصافية 100% — من غير فصحى رسمية جامدة، ومن غير أي لهجة خليجية (ممنوع تمامًا استخدام كلمات خليجية مثل: وش، شلون، أبغى، وايد، أو الفصحى المعقدة). تحدثي بأسلوب الشارع المصري الراقي والودود.

4. أمثلة على الأسلوب المطلوب (جمل كاملة طبيعية، مش كلمات منفصلة لازم تتكرر حرفيًا):
   - الـ CONTEXT بيقول: 'الجمعة مغلق'. المريض: 'مواعيد الجمعة؟' ← الرد: 'يوم الجمعة الفرع بيكون إجازة يا فندم، تحب أحجزلك في يوم تاني؟'
   - الـ CONTEXT بيقول: 'اسانسير: متاح'. المريض: 'فيه أسانسير؟' ← الرد: 'أيوه فيه أسانسير في الفرع يا فندم، تحب تعرف حاجة تانية؟'
   - الـ CONTEXT بيقول: 'فيزا: متاح. فاليو: غير متاح'. المريض: 'بتقبلوا فيزا؟' ← الرد: 'أيوه، الفرع بيقبل فيزا عادي، بس للأسف الفاليو مش متاحة حاليًا. حابب تعرف طريقة دفع تانية؟'

5. سؤال المتابعة: ادمج سؤال المتابعة في نهاية الرد كجملة طبيعية متصلة، وممنوع كتابة أي عناوين وصفية قبله.

6. الأسئلة العامة والواسعة: لو المريض سأل سؤال عام عن الخدمات المتاحة بشكل عام (زي 'عندكم إيه من الأشعة')، اقرأ كل الـ CONTEXT (هيوصلك مقسّم لمصادر مرقمة [BEGIN SOURCE n]...[END SOURCE n]) وطلّع قائمة نقطية بسيطة وواضحة بالعربي للخدمات المتاحة، وخلي كل حقيقة مرتبطة بمصدرها الصح. خليها مختصرة جدًا.

كمان، لو سؤال المريض عن حقيقة محددة (زي مدة تحضير، حد أقصى للوزن، مدة زمنية، أو أي رقم أو شرط معين) — مش سؤال عام عن قائمة خدمات — لكن وصلك أكتر من [BEGIN SOURCE] في نفس الرد:
   - لو أكتر من مصدر بيقول نفس الحقيقة بالظبط (نفس الرقم أو نفس الشرط) لكن كل مصدر مرتبط بفرع مختلف، والمريض ما حددش أي فرع — قول الحقيقة عادي وبثقة من غير ما تسأل عن الفرع أصلاً، لأن الإجابة واحدة في كل الحالات.
   - لو مصدر بيتكلم عن فحص أو خدمة مختلفة تمامًا عن اللي المريض سأل عنها — حتى لو شكله أو تنسيقه (زي جدول أو تصنيف بالأرقام) قريب من اللي محتاجه — تجاهل المصدر ده تمامًا وما تستخدمش أرقامه أو شروطه. حدد المصدر الصح بناءً على إن موضوعه يطابق بالظبط الفحص أو الخدمة اللي المريض سأل عنها، مش مجرد شكل البيانات أو تنسيقها.
   - ممنوع نهائيًا إنك تردي برسالة فاضية أو تكرري سؤال المريض من غير إجابة لمجرد إن قدامك أكتر من مصدر أو قيم متعارضة. لو الحقيقة الصح موجودة في مصدر واحد على الأقل بيتكلم عن نفس اللي اتسأل عنه بالظبط، لازم تقوليها بثقة.
"""

def generate(system, instruction, input_text):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"{instruction}\n{input_text}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    out = out[:, inputs.input_ids.shape[1]:]
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

import warnings
warnings.filterwarnings("ignore")

print(generate(
    system_prompt,
    "CONTEXT:\n[Document: Branch Directory] معلومات الفرع الإسعاف: متاح",
    "PATIENT MESSAGE:\nلو حصل طارئ وأنا في فرع حلوان، فيه عربية إسعاف موجودة؟",
))


Run this across a spread of real cases from `golden_test_suite.json`, including at
least one brand-filter cross-referral case (the specific scenario this redesign
targeted — e.g. the real MRA Abdomen/Technoscan case verified in `train.json`
during this session's dataset review) and one compound (2-field) narrow question.


## Stage 7 — Merge adapter into base model & push to Hugging Face

Safe to run in this SAME runtime, right after training — `llamafactory-cli export`
uses the already-installed LLaMA-Factory environment from Stage 1, no new or
conflicting packages (the torch/torchvision/torchaudio conflict that forces a
*separate* runtime only comes from vLLM's own install, which Stage 9 below still
needs its own fresh session for — see `merge_and_run_nilechat12b_v2.ipynb` if you'd
rather do that step in a standalone notebook instead).

Reads the adapter from the Hub repo `mennaharmas/raylab-nilechat-12b-v2-lora` (Stage 5's push target, not Drive) and pushes the merged full model to its own separate **private** repo, `mennaharmas/raylab-nilechat-12b-v2` — review Stage 6's real outputs above
before running this; merging + uploading a ~24GB model is not cheap to redo if the
checkpoint turns out to need another training pass.


> **Free the GPU before merging.** Stage 6 above loaded a full model (base +
> adapter) into GPU memory and never freed it — the export step below needs its own
> headroom to load and merge a fresh copy.


In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed")


In [ ]:
%%writefile /content/LLaMA-Factory/examples/merge_lora/raylab_merge_v2.yaml
### model
model_name_or_path: MBZUAI-Paris/Nile-Chat-12B
adapter_name_or_path: mennaharmas/raylab-nilechat-12b-v2-lora
template: gemma
trust_remote_code: true

### export
export_dir: /content/merged_model_v2/
export_size: 5
export_device: auto  # choices: [cpu, auto]
export_legacy_format: false


In [ ]:
!cd LLaMA-Factory && llamafactory-cli export examples/merge_lora/raylab_merge_v2.yaml


### Push the merged model to Hugging Face

Same proven pattern as `merge_and_run_nilechat12b_v2.ipynb` (`notebook_login` +
`create_repo` + `upload_folder`) — `private=True`. Token comes from the login
prompt below, never hardcoded (see that notebook's intro cell for why that matters
— the original v1 workflow had a real plaintext token leak here).


In [ ]:
from huggingface_hub import HfApi, notebook_login

# 1. Login -- a small box appears below this cell, paste your token (needs "Write"
#    scope) and click Login.
notebook_login()

# 2. Setup
api = HfApi()
repo_id = "mennaharmas/raylab-nilechat-12b-v2"

# 3. Create the repo (no-op if it already exists)
print("Creating repo on your account...")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=True)

# 4. Upload
print("Uploading merged model... (can take a while depending on Colab's network speed)")
api.upload_folder(
    folder_path="/content/merged_model_v2/",
    repo_id=repo_id,
    repo_type="model",
)
print(f"Model pushed to https://huggingface.co/{repo_id}")


## Stage 8 — Cost / throughput estimation

Real patient-query-shaped prompts, from the current golden suite (not the retired `golden_outputs_nilechat12b_1.json`).


In [ ]:
from datetime import datetime
import json

real_queries = [
    r["patient_query"]
    for r in json.load(open("golden_test_suite.json", encoding="utf-8"))
]

start = datetime.now()
input_tokens = output_tokens = 0

for q in real_queries[:30]:
    resp = generate(system_prompt, f"CONTEXT:\n", f"PATIENT MESSAGE:\n{q}")  # real retrieval context in production
    input_tokens += len(tokenizer.encode(q))
    output_tokens += len(tokenizer.encode(resp))

elapsed = (datetime.now() - start).total_seconds()
print(f"elapsed: {elapsed:.1f}s  tokens/sec: {(input_tokens + output_tokens) / elapsed:.1f}")


> **Free the GPU before the next stage needs it.**


In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed")


## Stage 9 — Serve with vLLM

Confirmed (Stage 11 below, carried forward unchanged from v1): `Gemma3ForCausalLM`
is vLLM-supported with LoRA support, and Nile-Chat-12B is text-only so the
vision-language integration gap (vLLM #14696) doesn't apply.


In [ ]:
!nohup vllm serve "MBZUAI-Paris/Nile-Chat-12B" \
  --dtype=bfloat16 --gpu-memory-utilization 0.85 \
  --max-lora-rank 16 --enable-lora \
  --lora-modules raylab-lora-v2="/gdrive/MyDrive/raylab-finetune-v2/models/" &

!tail -n 30 nohup.out


In [ ]:
import requests

r = requests.post("http://localhost:8000/v1/completions", json={
    "model": "raylab-lora-v2",
    "prompt": prompt,
    "max_tokens": 400,
    "temperature": 0.0,
})
print(r.json())


## Stage 10 — Load testing

Same substitution as Stage 8 -- real patient queries from the current golden suite.


In [ ]:
%%writefile locust.py
import json, random
from locust import HttpUser, task, between

real_queries = [
    r["patient_query"]
    for r in json.load(open("golden_test_suite.json", encoding="utf-8"))
]

class CompletionLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        message = {
            "model": "raylab-lora-v2",
            "prompt": random.choice(real_queries),
            "max_tokens": 400,
            "temperature": 0.0,
        }
        self.client.post("/v1/completions", json=message)


In [ ]:
!locust --headless -f locust.py --host=http://localhost:8000 -u 20 -r 1 -t "60s" --html=locust_results_v2.html


## Stage 11 — Verified values (carried forward from v1, unchanged)

Nothing about the base model, template, or serving stack changed in this redesign —
only the training data and the field-selection removal on the production side. All
four values below were confirmed against primary sources in the v1 notebook and
still apply as-is.

| # | Item | Confirmed value | Source |
|---|---|---|---|
| 1 | HF repo id | `MBZUAI-Paris/Nile-Chat-12B` | [model card](https://huggingface.co/MBZUAI-Paris/Nile-Chat-12B) + [config.json](https://huggingface.co/MBZUAI-Paris/Nile-Chat-12B/raw/main/config.json) (`architectures: ["Gemma3ForCausalLM"]`) |
| 2 | LLaMA-Factory `template` | `gemma` (not `gemma3` -- that registration wires vision-language `mm_plugin` support Nile-Chat-12B, text-only, doesn't ship a processor for) | [template.py](https://raw.githubusercontent.com/hiyouga/LLaMA-Factory/main/src/llamafactory/data/template.py) |
| 3 | `transformers` version range | `transformers>=4.55.0,<=5.8.0,!=4.57.0,!=5.6.0` | v4.50.0 release notes (Gemma3 floor) + LLaMA-Factory's own `check_dependencies()` ceiling |
| 4 | vLLM Gemma 3 support | Yes, incl. LoRA (`Gemma3ForCausalLM`) | [vLLM supported models](https://docs.vllm.ai/en/latest/models/supported_models/) |
